In [ ]:
import os
import shutil

repo_path = "/content/Flight-Delay-Forecasting"

# clone ONLY if not already present in this session
if not os.path.exists(repo_path):
    !git clone https://{GITHUB_TOKEN}@github.com/kevinbrugnera/Flight-Delay-Forecasting.git {repo_path}
    %cd {repo_path}
    !git checkout develop
    !git config --global user.email "your_email@example.com"
    !git config --global user.name "Your Name"
else:
    %cd {repo_path}
    !git pull origin develop  # pull any teammate changes before overwriting

# copy the updated notebook from Drive into the repo
filename = os.path.basename(NOTEBOOK_DRIVE_PATH)
shutil.copy(NOTEBOOK_DRIVE_PATH, os.path.join(repo_path, filename))

# commit and push
!git add {filename}
!git commit -m "{COMMIT_MESSAGE}"
!git push origin develop

print(f"\n✅ Done: {filename} uploaded to GitHub (develop branch)")

In [1]:
## 0. Setup (CPU-only: LightGBM doesn't need GPU)

!pip install -q lightgbm optuna shap pyarrow psutil

import gc, os, time
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
import shap
from sklearn.metrics import (roc_auc_score, f1_score, accuracy_score,
                              precision_score, recall_score, classification_report)
import pyarrow.parquet as pq
import psutil

optuna.logging.set_verbosity(optuna.logging.WARNING)

from google.colab import drive
drive.mount('/content/drive')

def checkpoint(label):
    rss = psutil.Process().memory_info().rss / 1024**2
    print(f"[{label}] RAM process: {rss:.0f} MB")

DATA_DIR = "/content/drive/MyDrive/aeolus_data/"
MODELS_DIR = DATA_DIR + "models/"
os.makedirs(MODELS_DIR, exist_ok=True)

n_threads = max(os.cpu_count() or 2, 1)
print(f"CPUs available: {n_threads}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CPUs available: 2


In [2]:
## 1. Loading data

def sample_parquet_streaming(path, frac=0.3, seed=42, batch_size=200_000):
    rng = np.random.default_rng(seed)
    pf = pq.ParquetFile(path)
    chunks = []
    for batch in pf.iter_batches(batch_size=batch_size):
        chunk = batch.to_pandas()
        sampled = chunk.sample(frac=frac, random_state=rng.integers(0, 1_000_000))
        chunks.append(sampled)
    return pd.concat(chunks, ignore_index=True)


train = sample_parquet_streaming(DATA_DIR + "train_encoded.parquet", frac=1)
val = pd.read_parquet(DATA_DIR + "val_encoded.parquet")
test = pd.read_parquet(DATA_DIR + "test_encoded.parquet")

target_col = "ARR_DELAY_BIN"
cat_cols = ["OP_CARRIER", "OP_CARRIER_FL_NUM", "FL_YEAR", "FL_MONTH", "FL_DAY", "FL_WEEK", "ORIGIN_INDEX", "DEST_INDEX"]
exclude_cols = {"ARR_DELAY", "DEP_DELAY", "ARR_DELAY_BIN", "DEP_DELAY_BIN", "_FLIGHT_DATE", "dep_hour_bucket"}
feature_cols = [c for c in train.columns if c not in exclude_cols]
lgbm_cat_cols = [c for c in cat_cols if c != "OP_CARRIER_FL_NUM"]  # hight cardinality

print(f"train: {train.shape}, val: {val.shape}, test: {test.shape}")
checkpoint("after loading")

train: (11389014, 38), val: (3868288, 38), test: (3868288, 38)
[after loading] RAM process: 4693 MB


In [3]:
## 2. Training baseline LightGBM

y_train, y_val, y_test = train[target_col], val[target_col], test[target_col]
scale_pos_weight = (len(y_train) - y_train.sum()) / max(y_train.sum(), 1)
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

train_set = lgb.Dataset(train[feature_cols], label=y_train, categorical_feature=lgbm_cat_cols)
val_set = lgb.Dataset(val[feature_cols], label=y_val, categorical_feature=lgbm_cat_cols, reference=train_set)

params = {
    "objective": "binary", "metric": ["auc", "binary_logloss"],
    "scale_pos_weight": scale_pos_weight, "learning_rate": 0.05,
    "num_leaves": 63, "max_bin": 63, "num_threads": n_threads, "verbose": -1,
}
model = lgb.train(
    params, train_set, num_boost_round=300,
    valid_sets=[train_set, val_set], valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(20), lgb.log_evaluation(50)],
)
gc.collect()

model.save_model(MODELS_DIR + "lgbm_baseline.txt")
print("Model saved on Drive.")
checkpoint("after training")

scale_pos_weight = 3.95
Training until validation scores don't improve for 20 rounds
Early stopping, best iteration is:
[1]	train's auc: 0.677335	train's binary_logloss: 0.49893	val's auc: 0.630219	val's binary_logloss: 0.478648
Model saved on Drive.
[after training] RAM process: 5384 MB


In [4]:
## 3. Optimal threshold + metrics on train/val/test

y_prob_train = model.predict(train[feature_cols], num_iteration=model.best_iteration)
y_prob_val = model.predict(val[feature_cols], num_iteration=model.best_iteration)
y_prob_test = model.predict(test[feature_cols], num_iteration=model.best_iteration)


def find_best_threshold(y_true, y_prob):
    thresholds = np.linspace(0.05, 0.95, 19)
    f1s = [f1_score(y_true, (y_prob > t).astype(int)) for t in thresholds]
    best_t = thresholds[int(np.argmax(f1s))]
    print(f"Optimal threshold (for val, max F1): {best_t:.2f}")
    return best_t


def metrics_from_probs(y_true, y_prob, threshold):
    y_pred = (y_prob > threshold).astype(int)
    return {
        "auc": roc_auc_score(y_true, y_prob), "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred), "precision": precision_score(y_true, y_pred),
        "recall": recall_score(y_true, y_pred),
    }


threshold = find_best_threshold(y_val, y_prob_val)
results = {
    "train": metrics_from_probs(y_train, y_prob_train, threshold),
    "val": metrics_from_probs(y_val, y_prob_val, threshold),
    "test": metrics_from_probs(y_test, y_prob_test, threshold),
}
report = pd.DataFrame(results).T
report.insert(0, "threshold", threshold)
print(report.round(4).to_string())

metrics = results["test"]
gap_auc = results["train"]["auc"] - results["test"]["auc"]
print(f"\nGap AUC train-test: {gap_auc:.4f} {'⚠️ possible overfitting' if gap_auc > 0.05 else '✅ ok'}")

report.to_csv(MODELS_DIR + "lgbm_baseline_metrics.csv")
checkpoint("after evaluation")

Optimal threshold (for val, max F1): 0.05
       threshold     auc  accuracy      f1  precision  recall
train       0.05  0.6773    0.2018  0.3359     0.2018     1.0
val         0.05  0.6302    0.1849  0.3121     0.1849     1.0
test        0.05  0.6590    0.1954  0.3270     0.1954     1.0

Gap AUC train-test: 0.0183 ✅ ok
[after evaluation] RAM process: 5545 MB


In [5]:
## 4. Feature importance

importance = pd.Series(model.feature_importance(importance_type="gain"), index=feature_cols).sort_values(ascending=False)
print(importance.head(15))
importance.to_csv(MODELS_DIR + "lgbm_feature_importance.csv")

CRS_DEP_TIME_MIN    1.029250e+06
OP_CARRIER          4.712425e+05
FL_WEEK             4.063444e+05
ORIGIN_INDEX        1.621626e+05
dep_hour_sin        1.573380e+05
O_PRCP              1.326598e+05
DEST_INDEX          7.896616e+04
D_PRCP              7.641225e+04
O_TEMP              5.822330e+04
arr_hour_cos        3.736846e+04
dow_sin             2.823458e+04
arr_hour_sin        1.574030e+04
dep_hour_cos        1.054490e+04
FL_MONTH            1.012170e+04
FL_DAY              9.401110e+03
dtype: float64


In [6]:
## 5. Evaluation for subgroup (month, airport)

def subgroup_report(group_col, top_n=None, min_support=30):
    tmp = test[[group_col, target_col]].copy()
    tmp["y_prob"] = y_prob_test
    tmp["y_pred"] = (y_prob_test > threshold).astype(int)
    rows = []
    for group_val, sub in tmp.groupby(group_col):
        if len(sub) < min_support or sub[target_col].nunique() < 2:
            continue
        rows.append({"group": group_val, "n": len(sub),
                      "auc": roc_auc_score(sub[target_col], sub["y_prob"]),
                      "f1": f1_score(sub[target_col], sub["y_pred"])})
    result = pd.DataFrame(rows).sort_values("n", ascending=False)
    return result.head(top_n).reset_index(drop=True) if top_n else result.reset_index(drop=True)

print(">>> For month")
print(subgroup_report("FL_MONTH"))
print("\n>>> Top 10 airports")
print(subgroup_report("ORIGIN_INDEX", top_n=10))

>>> For month
   group       n       auc        f1
0      9  603859  0.674278  0.448367
1      1  600153  0.598888  0.223982
2     10  596087  0.680257  0.370978
3      3  574089  0.600676  0.338577
4     11  571187  0.630837  0.258844
5      2  563258  0.592199  0.246155
6      8  359655  0.671126  0.397167

>>> Top 10 airports
   group       n       auc        f1
0     20  189118  0.674741  0.323645
1     84  175622  0.666946  0.386687
2     83  171151  0.655624  0.341407
3    224  156495  0.641432  0.369691
4     64  120358  0.672412  0.409740
5    173  108074  0.622357  0.285356
6    235  105006  0.652930  0.302371
7    171  104242  0.653972  0.336639
8    273   93758  0.599061  0.346113
9    179   87949  0.678462  0.308819


In [7]:
## 6. SHAP

sample = test[feature_cols].sample(n=min(2000, len(test)), random_state=42)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample)
mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_cols).sort_values(ascending=False)
print(mean_abs_shap.head(10))
del explainer, shap_values
gc.collect()

CRS_DEP_TIME_MIN    0.025514
FL_WEEK             0.015545
OP_CARRIER          0.014775
dep_hour_sin        0.007033
ORIGIN_INDEX        0.005983
O_PRCP              0.003294
DEST_INDEX          0.003284
D_PRCP              0.002425
arr_hour_cos        0.002131
dow_sin             0.001913
dtype: float64


/usr/local/lib/python3.12/dist-packages/shap/explainers/_tree.py:632: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


13

In [8]:
## 7. Hyperparameter tuning with Optuna

del train_set, val_set
gc.collect()
checkpoint("after freeing the main training datasets")

train_set_opt = lgb.Dataset(train[feature_cols], label=y_train, categorical_feature=lgbm_cat_cols,
                             params={"feature_pre_filter": False})
val_set_opt = lgb.Dataset(val[feature_cols], label=y_val, categorical_feature=lgbm_cat_cols,
                           reference=train_set_opt, params={"feature_pre_filter": False})


def objective(trial):
    p = {
        "objective": "binary", "metric": "auc", "scale_pos_weight": scale_pos_weight,
        "verbose": -1, "feature_pre_filter": False, "num_threads": n_threads, "max_bin": 63,
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 15, 127),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 200),
    }
    m = lgb.train(p, train_set_opt, num_boost_round=150, valid_sets=[val_set_opt],
                   callbacks=[lgb.early_stopping(15, verbose=False)])
    score = roc_auc_score(y_val, m.predict(val[feature_cols], num_iteration=m.best_iteration))
    del m
    gc.collect()
    return score

[after freeing the main training datasets] RAM process: 5429 MB


In [9]:
## 7a. Fast TEST with 3 trials

t0 = time.time()
study_test = optuna.create_study(direction="maximize")
study_test.optimize(objective, n_trials=3)
elapsed = time.time() - t0
print(f"3 trials: {elapsed:.1f}s -> estimate for 20 trials: {elapsed/3*20/60:.1f} minutes")
checkpoint("after test Optuna")

3 trials: 1359.5s -> estimate for 20 trials: 151.1 minutes
[after test Optuna] RAM process: 5841 MB


In [10]:
## 7b. Real tuning

import joblib

study = optuna.create_study(direction="maximize")

def save_callback(study, trial):
    joblib.dump(study, MODELS_DIR + "optuna_study_lgbm.pkl")

study.optimize(objective, n_trials=20, callbacks=[save_callback], show_progress_bar=True)
print("Best AUC:", study.best_value, "Params:", study.best_params)
checkpoint("after full tuning")

  0%|          | 0/20 [00:00<?, ?it/s]

Best AUC: 0.6604520899728623 Params: {'learning_rate': 0.10480532067775375, 'num_leaves': 93, 'min_child_samples': 131}
[after full tuning] RAM process: 5842 MB


In [11]:
## 8. Final retraining with best hyperparameters + evaluation

best_params = {
    "objective": "binary", "metric": ["auc", "binary_logloss"], "scale_pos_weight": scale_pos_weight,
    "verbose": -1, "feature_pre_filter": False, "num_threads": n_threads, "max_bin": 63,
    **study.best_params,
}
model_tuned = lgb.train(
    best_params, train_set_opt, num_boost_round=500,
    valid_sets=[train_set_opt, val_set_opt], valid_names=["train", "val"],
    callbacks=[lgb.early_stopping(30), lgb.log_evaluation(50)],
)
model_tuned.save_model(MODELS_DIR + "lgbm_tuned.txt")

y_prob_train_t = model_tuned.predict(train[feature_cols], num_iteration=model_tuned.best_iteration)
y_prob_val_t = model_tuned.predict(val[feature_cols], num_iteration=model_tuned.best_iteration)
y_prob_test_t = model_tuned.predict(test[feature_cols], num_iteration=model_tuned.best_iteration)

threshold_tuned = find_best_threshold(y_val, y_prob_val_t)
results_tuned = {
    "train": metrics_from_probs(y_train, y_prob_train_t, threshold_tuned),
    "val": metrics_from_probs(y_val, y_prob_val_t, threshold_tuned),
    "test": metrics_from_probs(y_test, y_prob_test_t, threshold_tuned),
}
report_tuned = pd.DataFrame(results_tuned).T
print(report_tuned.round(4).to_string())

metrics_tuned = results_tuned["test"]
report_tuned.to_csv(MODELS_DIR + "lgbm_tuned_metrics.csv")
print(f"\nComparison AUC test — baseline: {metrics['auc']:.4f} | tunated: {metrics_tuned['auc']:.4f}")
checkpoint("final")

Training until validation scores don't improve for 30 rounds
Early stopping, best iteration is:
[1]	train's auc: 0.681712	train's binary_logloss: 0.49591	val's auc: 0.631913	val's binary_logloss: 0.479047
Optimal threshold (for val, max F1): 0.05
          auc  accuracy      f1  precision  recall
train  0.6817    0.2018  0.3359     0.2018     1.0
val    0.6319    0.1849  0.3121     0.1849     1.0
test   0.6614    0.1954  0.3270     0.1954     1.0

Comparison AUC test — baseline: 0.6590 | tunated: 0.6614
[final] RAM process: 5930 MB
